# 🚀 Huấn Luyện Mô Hình Deep Learning Sửa Lỗi Chính Tả Tiếng Việt (ViT5 Transformer)
## 📚 Bộ Dữ Liệu: VSEC Corpus (Vietnamese Spelling Error Correction - 9.341 câu, 11.202 lỗi thực tế)
**Môn học**: Học sâu (3 tín chỉ) - Đồ án: Phát hiện & Tự động sửa lỗi chính tả tiếng Việt

> **Lưu ý quan trọng**: Hãy vào menu **Runtime > Change runtime type** (Thời gian chạy > Thay đổi loại thời gian chạy) và chọn **T4 GPU** để huấn luyện siêu tốc (~8-10 phút)!

In [ ]:
# 1. Kiểm tra phần cứng GPU Tesla T4
!nvidia-smi

In [ ]:
# 2. Cài đặt thư viện chuyên dụng tương thích 100% với Colab
!pip install -q "transformers>=4.38.0,<4.46.0" datasets accelerate sentencepiece evaluate jiwer rapidfuzz

In [ ]:
import os
import json
import random
import urllib.request
import numpy as np
import torch
from datasets import Dataset
import evaluate

# 3. Vá lỗi tương thích Unigram Sequence/dict trong thư viện T5 Tokenizer
try:
    from transformers.models.t5 import tokenization_t5
    _orig_t5_init = tokenization_t5.T5Tokenizer.__init__

    def _patched_t5_init(self, *args, **kwargs):
        if "vocab" in kwargs and isinstance(kwargs["vocab"], dict):
            kwargs["vocab"] = [(k, float(v)) for k, v in kwargs["vocab"].items()]
        elif len(args) > 0 and isinstance(args[0], dict):
            args = ([(k, float(v)) for k, v in args[0].items()],) + args[1:]
        return _orig_t5_init(self, *args, **kwargs)

    tokenization_t5.T5Tokenizer.__init__ = _patched_t5_init
except Exception:
    pass

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Đang huấn luyện trên thiết bị: {device}")

# Tải mô hình ViT5 và Tokenizer từ HuggingFace
MODEL_NAME = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
print("[*] Đã nạp thành công mô hình ViT5!")

In [ ]:
# 4. Tải và xử lý toàn bộ Bộ dữ liệu chuẩn VSEC Corpus (9.341 câu - 11.202 lỗi thực tế)
VSEC_URL = "https://raw.githubusercontent.com/VSEC2021/VSEC/main/Dataset/VSEC.jsonl"
VSEC_FILE = "VSEC.jsonl"

if not os.path.exists(VSEC_FILE):
    print("[*] Đang tải VSEC Corpus từ GitHub...")
    urllib.request.urlretrieve(VSEC_URL, VSEC_FILE)
    print(f"[*] Tải xong file {VSEC_FILE} ({os.path.getsize(VSEC_FILE) / (1024*1024):.1f} MB)")
else:
    print(f"[*] Đã có sẵn file {VSEC_FILE}")

# Đọc và trích xuất cặp (input_text, target_text)
dataset_records = []
with open(VSEC_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            item = json.loads(line)
        except Exception:
            continue
        ann = item.get("annotations", [])
        if not ann:
            continue

        inp_tokens = []
        tgt_tokens = []
        for a in ann:
            curr = a.get("current_syllable", "")
            alt = a.get("alternative_syllables", [])
            is_corr = a.get("is_correct", True)
            inp_tokens.append(curr)
            if not is_corr and alt:
                tgt_tokens.append(alt[0])
            else:
                tgt_tokens.append(curr)

        inp_sent = " ".join(inp_tokens).strip()
        tgt_sent = " ".join(tgt_tokens).strip()
        if inp_sent and tgt_sent:
            dataset_records.append({
                "input_text": f"sửa lỗi: {inp_sent}",
                "target_text": tgt_sent
            })

print(f"[*] Đã trích xuất thành công {len(dataset_records)} mẫu câu từ VSEC Corpus!")

# Chia tập Train (90%) và Validation (10%)
random.seed(42)
random.shuffle(dataset_records)
split_idx = int(len(dataset_records) * 0.9)

train_dataset = Dataset.from_list(dataset_records[:split_idx])
val_dataset = Dataset.from_list(dataset_records[split_idx:])

print(f"[*] Số mẫu Train     : {len(train_dataset)}")
print(f"[*] Số mẫu Validation: {len(val_dataset)}")
print(f"[*] Ví dụ mẫu 1:")
print(f"    Input : {dataset_records[0]['input_text']}")
print(f"    Target: {dataset_records[0]['target_text']}")

In [ ]:
# 5. Tiền xử lý Tokenize dữ liệu cho ViT5
def preprocess_function(examples):
    inputs = examples["input_text"]
    targets = examples["target_text"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("[*] Đang mã hóa dữ liệu Tokenizer...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)
print("[*] Đã mã hóa xong toàn bộ dữ liệu Train & Validation!")

In [ ]:
# 6. Thiết lập tham số huấn luyện chuẩn cho GPU Tesla T4
training_args = Seq2SeqTrainingArguments(
    output_dir="./vit5_vsec_checkpoints",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
)

print("[*] Bắt đầu huấn luyện ViT5 trên toàn bộ 9.341 câu VSEC Corpus (GPU Tesla T4)...")
trainer.train()
print("[*] Huấn luyện hoàn tất thành công!")

In [ ]:
# 7. Thử nghiệm thực tế mô hình vừa huấn luyện trên các câu khó / lỗi ngữ cảnh
test_cases = [
    "tôi đang hoc môj họt sâu",
    "Thời tiết miền Bắc mùa đông se lạnh va xó mu7a phùn lhẹ.",
    "Tối nay tôi tự nấu ăn ở nhà để tiet kiệm fiền",
    "Tôi đi học về và thấy mẹy đang nấu cơm",
    "gìn giữ môi trường sanh , sạch , đẹp",
    "Giữ được chỉ số này không giam mà còn tăng thêm"
]

print("=" * 60)
print("KIỂM TRA THỰC TẾ TRÊN MÔ HÌNH VỪA TRAIN XONG (VSEC CORPUS):")
print("=" * 60)
model.eval()
for sent in test_cases:
    inp = tokenizer(f"sửa lỗi: {sent}", return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_length=128,
            no_repeat_ngram_size=4,
            repetition_penalty=1.2
        )
    corrected = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"❌ Câu gốc: {sent}")
    print(f"✅ Sửa lại: {corrected}")
    print("-" * 60)

In [ ]:
# 8. Xuất mô hình tốt nhất và tự động tải file zip về máy tính
save_path = "./best_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"[*] Đã lưu mô hình tốt nhất vào: {save_path}")

# Nén file để tải về
!zip -r vit5_best_model.zip best_model
print("[*] Đã nén thành công vit5_best_model.zip!")

# Tự động kích hoạt tải file về máy tính qua trình duyệt
try:
    from google.colab import files
    files.download("vit5_best_model.zip")
    print("[*] Đang tự động tải vit5_best_model.zip về máy tính của bạn...")
except Exception as e:
    print("[*] Bạn hãy bấm vào biểu tượng thư mục ở thanh bên trái Colab, chuột phải vào vit5_best_model.zip và chọn Download!")